In [1]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from utils import getMatchAndPlayerStats,regular_games_total,regular_season_all_parts
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
pd.set_option('future.no_silent_downcasting', True)


gameP = getMatchAndPlayerStats(regular_games_total, regular_season_all_parts)

# === DEFINIR FEATURES E TARGET ===

X = gameP.drop(['personName', 'teamTricode', 'season_year', 'plusMinusPoints', 'winPercentage'], axis=1)
y = gameP['plusMinusPoints']

# === NORMALIZAR ===

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === SPLIT ===

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)



# === FEATURE SELECTION COM RANDOM FOREST ===

selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42))
selector.fit(X_train, y_train)

X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

selected_features = np.array(X.columns)[selector.get_support()]
print("Selected features:", list(selected_features))


# === VANILLA XGBOOST ===

xgb_vanilla = XGBRegressor(n_estimators=100)
xgb_vanilla.fit(X_train_sel, y_train)
y_pred_xgb_vanilla = xgb_vanilla.predict(X_test_sel)
print("\n Vanilla XGBoost:")
print("R2:", r2_score(y_test, y_pred_xgb_vanilla))
print("MSE:", mean_squared_error(y_test, y_pred_xgb_vanilla))


# === TUNED XGBOOST ===

param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(),
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_xgb.fit(X_train_sel, y_train)
best_xgb = grid_search_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_sel)

print("\n Tuned XGBoost:")
print("Best params:", grid_search_xgb.best_params_)
print("R2:", r2_score(y_test, y_pred_xgb))
print("MSE:", mean_squared_error(y_test, y_pred_xgb))


# === VANILLA RANDOM FOREST ===

rf_vanilla = RandomForestRegressor(n_estimators=100, random_state=42)
rf_vanilla.fit(X_train_sel, y_train)
y_pred_rf_vanilla = rf_vanilla.predict(X_test_sel)

print("\n Vanilla Random Forest:")
print("R2:", r2_score(y_test, y_pred_rf_vanilla))
print("MSE:", mean_squared_error(y_test, y_pred_rf_vanilla))


# === TUNED RANDOM FOREST ===

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 25, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [2, 5],
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_rf.fit(X_train_sel, y_train)
best_rf = grid_search_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_sel)

print("\n Tuned Random Forest:")
print("Best params:", grid_search_rf.best_params_)
print("R2:", r2_score(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))

Selected features: ['WL', 'minutesParsed', 'gamesPlayed']

 Vanilla XGBoost:
R2: 0.44215162227352
MSE: 5.2831426234621395
Fitting 3 folds for each of 16 candidates, totalling 48 fits

 Tuned XGBoost:
Best params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
R2: 0.4994944007259192
MSE: 4.740073773420315

 Vanilla Random Forest:
R2: 0.41256626759894943
MSE: 5.563332823079618
Fitting 3 folds for each of 32 candidates, totalling 96 fits

 Tuned Random Forest:
Best params: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
R2: 0.48249623413267984
MSE: 4.901056115639271
